# 中音谱号个人 Logo 探索

以**中音谱号（C 谱号）**为母题：左侧一粗一细两条竖线，右侧上下两瓣弧线，在**第三线 y = 0** 处收束成指向竖线的“喙”——
这个收束点就是中音谱号标定中央 C 的位置。三种方案共享同一组基础笔画（`clef.base_strokes`），只在“如何把骨架变成图形”上不同：

| 方案 | 思路 |
|---|---|
| A 圆弧线描 | 等宽圆头线 + 弧瓣末端圆点，细竖线用点缀色 |
| B 实心切割 | 方头实心竖线；弧瓣 = 实心椭圆减去内椭圆的月牙，喙为尖角 |
| C 谱线融合 | 在 A 的笔画上叠加五线谱，谱线避让笔画，第三线从喙点向右伸出 |

坐标约定：线距 = `height/4`，五条线在 `y = k·height/4 (k=-2..2)`。所有几何在 `clef.py`，本 notebook 只负责参数、展示与导出；
从项目根目录顺序执行即可，无需交互。配色借鉴 Material 的“主色 + 单一强调色 + 中性底”思路，但不使用其色板。

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

import clef

## 集中参数

只改这里，下面所有图都随之更新。取值范围由 `clef.Params` 校验，越界会直接抛 `ValueError`：

- `HEIGHT`：竖线总高（= 五线谱上下线距离），`> 0`
- `WIDTH`：整体宽度，`[min_width, HEIGHT]`
- `WEIGHT`：基础笔画粗细（粗竖线为 2 倍），`(0, HEIGHT/8]`
- `OPENING`：弧瓣末端开合，`[0, 0.8]`，0 最闭合
- `GAP`：竖线之间、细竖线与弧瓣之间的间距，`[0, HEIGHT/4]`

In [ ]:
HEIGHT = 4.0
WIDTH = 3.0
WEIGHT = 0.3
OPENING = 0.25
GAP = 0.3
PALETTE = "ink"  # 导出用配色：ink / teal / cobalt

p = clef.Params(height=HEIGHT, width=WIDTH, weight=WEIGHT, opening=OPENING, gap=GAP)
print(p)
print(f"喙点 x0 = {p.x0:.3f}，当前粗细/间距下最小宽度 = {p.min_width():.3f}")
for key, pal in clef.PALETTES.items():
    print(f"{key:7s} {pal.name}: fg={pal.fg} accent={pal.accent} bg={pal.bg}")

## 参考构形：五线谱上的位置

把方案 A 放回五线谱：两瓣在第三线（红线）收束，喙点（圆点）指向竖线。这是三种方案都必须保留的骨架关系。

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.2))
pal = clef.PALETTES["ink"]
for k in range(-2, 3):
    y = k * p.height / 4
    ax.plot([-1, p.width + 1.5], [y, y], color=pal.accent if k == 0 else "#8A8F98", lw=1.6 if k == 0 else 0.9, zorder=0)
    ax.text(p.width + 1.8, y, f"line {k + 3}" + (" = middle C" if k == 0 else ""), va="center", fontsize=9)
for s in clef.variant_a(p):
    ax.fill(*s.poly.T, color=pal.fg if s.role == "fg" else pal.accent, lw=0, zorder=2)
ax.plot([p.x0], [0], "o", color=pal.accent, mec="white", ms=7, zorder=3)
ax.set_xlim(-1.5, p.width + 5.5)
ax.set_ylim(-p.height * 0.75, p.height * 0.75)
ax.set_aspect("equal")
ax.axis("off")
plt.show()

## 逐笔拆解

白色虚线是共享骨架（中心线），深色为方案 A 的轮廓；最后一格组合并标出第三线与喙点。下弧瓣严格是上弧瓣关于 y = 0 的镜像。

In [ ]:
display(clef.breakdown_figure(p, PALETTE))

## 三方案同尺度

三张图使用同一个取景框（`clef.common_bbox`），大小和位置可直接比较。

In [ ]:
bbox = clef.common_bbox(p)
fig, axes = plt.subplots(1, 3, figsize=(10, 3.6))
for ax, (label, fn) in zip(axes, clef.VARIANTS.values()):
    clef.draw_shapes(ax, fn(p), PALETTE, bbox)
    ax.set_title(label)
plt.show()

## 三方案 × 三配色 + 单色

列依次为：墨蓝·朱红·米白（ink）、深青·琥珀·暖白（teal）、炭黑·钴蓝·白（cobalt）、单色。单色列用来检查去掉颜色后结构是否仍成立。

In [ ]:
display(clef.compare_figure(p))

## 参数小扫描

每次只改一个参数，其余沿用上面的集中参数；每格按该参数组合自动取景，便于比较形状，不用于比较绝对尺寸。若某个取值越界，会在格子里显示原因而不是中断执行。

In [ ]:
sweeps = {
    "opening": [0.0, 0.25, 0.5, clef.OPENING_MAX],
    "weight": [0.2, 0.3, 0.4, 0.5],
    "gap": [0.0, 0.15, 0.3, 0.5],
    "width": [2.4, 3.0, 3.5, 4.0],
}
base = dict(height=p.height, width=p.width, weight=p.weight, opening=p.opening, gap=p.gap)
for variant in "abc":
    label, fn = clef.VARIANTS[variant]
    fig, axes = plt.subplots(len(sweeps), 4, figsize=(9, 9))
    fig.suptitle(label)
    for row, (name, values) in zip(axes, sweeps.items()):
        for ax, v in zip(row, values):
            try:
                q = clef.Params(**{**base, name: v})
            except ValueError as err:
                print(f"{label} {name}={v:g} 越界：{err}")
                ax.text(0.5, 0.5, f"{name}={v:g}\nout of range", ha="center", va="center", fontsize=8)
                ax.axis("off")
                continue
            clef.draw_shapes(ax, fn(q), PALETTE, clef.common_bbox(q))
            ax.set_title(f"{name}={v:g}", fontsize=9)
    fig.tight_layout()
    plt.show()

### 边界校验

下面几个参数组合应当被拒绝。

In [ ]:
for bad in [dict(weight=0), dict(opening=0.9), dict(gap=-0.1), dict(width=1.0), dict(width=4.5), dict(height=float("nan"))]:
    try:
        clef.Params(**{**base, **bad})
        print(bad, "-> 意外通过")
    except ValueError as err:
        print(bad, "->", err)

## 小尺寸与大图

32 px / 64 px 为按真实像素栅格化后再放大显示（最近邻插值），用来人工观察小尺寸下笔画是否糊在一起；
自动测试只检查墨迹存在与上下对称，不判断辨识度。

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(8, 5.6))
for col, (label, fn) in enumerate(clef.VARIANTS.values()):
    for row, px in enumerate((32, 64)):
        axes[row, col].imshow(clef.rasterize(fn(p), PALETTE, px), interpolation="nearest")
        axes[row, col].set_title(f"{label} @ {px}px", fontsize=9)
        axes[row, col].axis("off")
plt.show()

for label, fn in clef.VARIANTS.values():
    display(clef.render(fn(p), PALETTE, size=5))

## 导出

写出 `out/variant_{a,b,c}.{png,svg}`、`out/compare.png`、`out/strokes_breakdown.png`（`out/` 已被 git 忽略）。
命令行等价操作：`uv run python clef.py`（使用默认参数）。

In [ ]:
written = clef.export_all("out", p, PALETTE)
for name, path in written.items():
    print(f"{name:24s} {path.stat().st_size:>8d} bytes")